In [1]:
RUN_ID = "full_reference_r1"     # the run whose saved model you are checking
 
import findspark; findspark.init("/usr/lib/spark")
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName("task5_reload_check").master("yarn")
         .enableHiveSupport().getOrCreate())
OUT_BASE = f"hdfs:///project/runs/{RUN_ID}/task3"     # HDFS, NOT the laptop folder task3_outputs/
print("fresh application:", spark.sparkContext.applicationId)
print("checking model under:", OUT_BASE)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 08:19:05 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


fresh application: application_1794292319670_0018
checking model under: hdfs:///project/runs/full_reference_r1/task3


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51470)
Traceback (most recent call last):
  File "/usr/lib64/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib64/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib64/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib64/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_int(self.rfile)
  File "/usr/lib/spark/python/pyspark/

In [2]:
# (Her full Section 0 is NOT pasted: it would reset OUT_BASE to the old laptop folder and
#  expects TASK2_EXPECTED_ROWS, which only exists in the pipeline notebook.)
import json, datetime
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
 
MODEL_PATH    = OUT_BASE + "/model_best"                   # saved final model
FEATURES_PATH = OUT_BASE + "/weekly_features"              # the test features that run used
REF_PRED_PATH = OUT_BASE + "/reference_test_predictions"   # that run's predictions (the "answer key")
EVIDENCE_PATH = OUT_BASE + "/evidence"
TOL = 1e-9                                                 # same tolerance as Felicia's
 
def save_json(obj, path):
    spark.createDataFrame([(json.dumps(obj, indent=2, default=str),)], ["json"]).coalesce(1).write.mode("overwrite").text(path)
 
# Show the files exist on HDFS before loading (evidence that nothing is missing)
import subprocess
print(subprocess.run(f"hadoop fs -ls {MODEL_PATH} {MODEL_PATH}/stages", shell=True,
                     text=True, capture_output=True).stdout)
 

Found 2 items
drwxr-xr-x   - hadoop hdfsadmingroup          0 2026-11-10 08:02 hdfs:///project/runs/full_reference_r1/task3/model_best/metadata
drwxr-xr-x   - hadoop hdfsadmingroup          0 2026-11-10 08:02 hdfs:///project/runs/full_reference_r1/task3/model_best/stages
Found 4 items
drwxr-xr-x   - hadoop hdfsadmingroup          0 2026-11-10 08:02 hdfs:///project/runs/full_reference_r1/task3/model_best/stages/0_Imputer_99d9eae6de06
drwxr-xr-x   - hadoop hdfsadmingroup          0 2026-11-10 08:02 hdfs:///project/runs/full_reference_r1/task3/model_best/stages/1_VectorAssembler_abf21c526ea3
drwxr-xr-x   - hadoop hdfsadmingroup          0 2026-11-10 08:02 hdfs:///project/runs/full_reference_r1/task3/model_best/stages/2_StandardScaler_9681b400a533
drwxr-xr-x   - hadoop hdfsadmingroup          0 2026-11-10 08:02 hdfs:///project/runs/full_reference_r1/task3/model_best/stages/3_LinearRegression_41a519653f8e



In [3]:
fresh = SparkSession.builder.getOrCreate()
 
# Which Spark version saved the model? (A different major.minor version may be incompatible.)
saved_spark_version = json.loads(spark.read.text(MODEL_PATH + "/metadata").first()[0]).get("sparkVersion")
print("model saved with Spark", saved_spark_version, "| loading with Spark", spark.version)
if saved_spark_version and saved_spark_version.split(".")[:2] != spark.version.split(".")[:2]:
    print("WARNING: Spark major.minor version differs from the one that saved the model; loading may fail or give different results.")
 
# Load the model without retraining.
model2 = PipelineModel.load(MODEL_PATH)                       # fails loudly if files are missing/incompatible
 
# Run the reloaded model on the same test rows and compute new predicted volatility (p_new).
feat = spark.read.parquet(FEATURES_PATH).filter("status = 'ok' AND period = 'test'")
new = (model2.transform(feat)
             .select("symbol", "anchor_date", F.col("prediction").alias("p_new")))
 
# Load the original predictions (p_ref).
ref = spark.read.parquet(REF_PRED_PATH).select("symbol", "anchor_date", F.col("prediction").alias("p_ref"))
 
# Join the new predictions and the reference predictions by stock and date.
cmp = new.join(ref, ["symbol", "anchor_date"], "full_outer")
 
st = cmp.agg(F.count("*").alias("n"),
             F.sum(F.when(F.col("p_new").isNull() | F.col("p_ref").isNull(), 1).otherwise(0)).alias("unmatched_rows"),
             F.max(F.abs(F.col("p_new") - F.col("p_ref"))).alias("max_abs_diff")).first()
 
passed = (st["unmatched_rows"] == 0) and (st["max_abs_diff"] is not None) and (st["max_abs_diff"] <= TOL)
record = {"check": "reload saved PipelineModel in fresh session and reproduce test predictions",
          "command": "PipelineModel.load(MODEL_PATH).transform(stored test features)", "tolerance": TOL,
          "rows_compared": st["n"], "unmatched_rows": st["unmatched_rows"], "max_abs_pred_diff": st["max_abs_diff"],
          "result": "PASS" if passed else "FAIL",
          "spark_version": spark.version, "spark_version_saved": saved_spark_version,
          "fresh_application_id": spark.sparkContext.applicationId,           # added: proves a new session
          "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "model_path": MODEL_PATH}
save_json(record, EVIDENCE_PATH + "/task3_reload_check")
print(json.dumps(record, indent=2, default=str))
 


model saved with Spark 3.5.5-amzn-1 | loading with Spark 3.5.5-amzn-1


{
  "check": "reload saved PipelineModel in fresh session and reproduce test predictions",
  "command": "PipelineModel.load(MODEL_PATH).transform(stored test features)",
  "tolerance": 1e-09,
  "rows_compared": 25441,
  "unmatched_rows": 0,
  "max_abs_pred_diff": 0.0,
  "result": "PASS",
  "spark_version": "3.5.5-amzn-1",
  "spark_version_saved": "3.5.5-amzn-1",
  "fresh_application_id": "application_1794292319670_0018",
  "timestamp_utc": "2026-11-10T08:19:54.513640+00:00",
  "model_path": "hdfs:///project/runs/full_reference_r1/task3/model_best"
}
